# 004 · Gold: modelo dimensional (Ralph Kimball)

**Trabalho 3 — Lakehouse com Databricks e arquitetura medalhão** · Autores: **Murilo Mandelli** e **Augusto Benedetti**

Lê a **silver** e alimenta um **esquema estrela** no schema **`gold`**, pronto para análise.

| Tabela | Tipo | Grão / conteúdo | Chave |
|---|---|---|---|
| `dim_cliente` | Dimensão | Um registro por cliente | `SK_CLIENTE` (identity) |
| `dim_produto` | Dimensão | Um registro por produto, **com a categoria desnormalizada** | `SK_PRODUTO` (identity) |
| `dim_tempo` | Dimensão | Um registro por dia (2024–2026) | `DATA` |
| `fato_vendas` | Fato | **Um registro por item vendido** | FKs para as 3 dimensões |

As dimensões usam **chave substituta** (`SK_`, gerada pelo Delta com `GENERATED BY DEFAULT AS
IDENTITY`) e são carregadas com **MERGE** (SCD Tipo 1: atualiza quem mudou, insere quem é novo) —
mesmo padrão do notebook da aula. O número do pedido e o status ficam no fato como **dimensões
degeneradas**.

In [ ]:
for tabela in ["categoria", "cliente", "produto", "pedido", "item_pedido"]:
    spark.table(f"silver.{tabela}").createOrReplaceTempView(tabela)
print("Views temporárias criadas a partir da silver.")

## dim_cliente

In [ ]:
%sql
DROP TABLE IF EXISTS gold.dim_cliente

In [ ]:
%sql
CREATE TABLE gold.dim_cliente (
    SK_CLIENTE          BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_CLIENTE      INT,
    NOME                STRING,
    EMAIL               STRING,
    CPF                 STRING,
    CIDADE              STRING,
    UNIDADE_FEDERATIVA  STRING,
    DATA_CADASTRO       DATE
) USING DELTA

In [ ]:
%sql
MERGE INTO gold.dim_cliente AS d
USING (
    SELECT CODIGO_CLIENTE, NOME, EMAIL, CPF, CIDADE, UNIDADE_FEDERATIVA, DATA_CADASTRO FROM cliente
) AS s
ON d.CODIGO_CLIENTE = s.CODIGO_CLIENTE
WHEN MATCHED AND (d.NOME <> s.NOME OR d.EMAIL <> s.EMAIL OR d.CIDADE <> s.CIDADE
                  OR d.UNIDADE_FEDERATIVA <> s.UNIDADE_FEDERATIVA) THEN
    UPDATE SET NOME = s.NOME, EMAIL = s.EMAIL, CPF = s.CPF, CIDADE = s.CIDADE,
               UNIDADE_FEDERATIVA = s.UNIDADE_FEDERATIVA, DATA_CADASTRO = s.DATA_CADASTRO
WHEN NOT MATCHED THEN
    INSERT (CODIGO_CLIENTE, NOME, EMAIL, CPF, CIDADE, UNIDADE_FEDERATIVA, DATA_CADASTRO)
    VALUES (s.CODIGO_CLIENTE, s.NOME, s.EMAIL, s.CPF, s.CIDADE, s.UNIDADE_FEDERATIVA, s.DATA_CADASTRO)

In [ ]:
%sql
SELECT * FROM gold.dim_cliente ORDER BY SK_CLIENTE LIMIT 5

## dim_produto (com a categoria desnormalizada)

In [ ]:
%sql
DROP TABLE IF EXISTS gold.dim_produto

In [ ]:
%sql
CREATE TABLE gold.dim_produto (
    SK_PRODUTO           BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_PRODUTO       INT,
    NOME_PRODUTO         STRING,
    CATEGORIA            STRING,
    DESCRICAO_CATEGORIA  STRING,
    PRECO_ATUAL          DECIMAL(10,2),
    ATIVO                BOOLEAN
) USING DELTA

In [ ]:
%sql
MERGE INTO gold.dim_produto AS d
USING (
    SELECT p.CODIGO_PRODUTO, p.NOME AS NOME_PRODUTO, c.NOME AS CATEGORIA,
           c.DESCRICAO AS DESCRICAO_CATEGORIA, CAST(p.PRECO AS DECIMAL(10,2)) AS PRECO_ATUAL,
           CAST(p.ATIVO AS BOOLEAN) AS ATIVO
    FROM produto p
    INNER JOIN categoria c ON c.CODIGO_CATEGORIA = p.CODIGO_CATEGORIA
) AS s
ON d.CODIGO_PRODUTO = s.CODIGO_PRODUTO
WHEN MATCHED AND (d.NOME_PRODUTO <> s.NOME_PRODUTO OR d.CATEGORIA <> s.CATEGORIA
                  OR d.PRECO_ATUAL <> s.PRECO_ATUAL OR d.ATIVO <> s.ATIVO) THEN
    UPDATE SET NOME_PRODUTO = s.NOME_PRODUTO, CATEGORIA = s.CATEGORIA,
               DESCRICAO_CATEGORIA = s.DESCRICAO_CATEGORIA, PRECO_ATUAL = s.PRECO_ATUAL, ATIVO = s.ATIVO
WHEN NOT MATCHED THEN
    INSERT (CODIGO_PRODUTO, NOME_PRODUTO, CATEGORIA, DESCRICAO_CATEGORIA, PRECO_ATUAL, ATIVO)
    VALUES (s.CODIGO_PRODUTO, s.NOME_PRODUTO, s.CATEGORIA, s.DESCRICAO_CATEGORIA, s.PRECO_ATUAL, s.ATIVO)

In [ ]:
%sql
SELECT * FROM gold.dim_produto ORDER BY SK_PRODUTO LIMIT 5

## dim_tempo

In [ ]:
%sql
DROP TABLE IF EXISTS gold.dim_tempo

In [ ]:
data_inicial, data_final = "2024-01-01", "2026-12-31"
num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

df_tempo = (
    spark.range(0, num_dias + 1)
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS DATA")
    .selectExpr(
        "DATA",
        "year(DATA) AS ANO",
        "quarter(DATA) AS TRIMESTRE",
        "month(DATA) AS MES",
        """CASE month(DATA) WHEN 1 THEN 'JANEIRO' WHEN 2 THEN 'FEVEREIRO' WHEN 3 THEN 'MARCO'
           WHEN 4 THEN 'ABRIL' WHEN 5 THEN 'MAIO' WHEN 6 THEN 'JUNHO' WHEN 7 THEN 'JULHO'
           WHEN 8 THEN 'AGOSTO' WHEN 9 THEN 'SETEMBRO' WHEN 10 THEN 'OUTUBRO'
           WHEN 11 THEN 'NOVEMBRO' WHEN 12 THEN 'DEZEMBRO' END AS NOME_MES""",
        "day(DATA) AS DIA",
        """CASE dayofweek(DATA) WHEN 1 THEN 'DOMINGO' WHEN 2 THEN 'SEGUNDA-FEIRA'
           WHEN 3 THEN 'TERCA-FEIRA' WHEN 4 THEN 'QUARTA-FEIRA' WHEN 5 THEN 'QUINTA-FEIRA'
           WHEN 6 THEN 'SEXTA-FEIRA' WHEN 7 THEN 'SABADO' END AS NOME_DIA_SEMANA""",
        "dayofweek(DATA) AS NUMERO_DIA_SEMANA",
    )
)
df_tempo.write.format("delta").mode("overwrite").saveAsTable("gold.dim_tempo")
print("gold.dim_tempo:", df_tempo.count(), "dias")

## fato_vendas (grão: um item vendido)

In [ ]:
%sql
DROP TABLE IF EXISTS gold.fato_vendas

In [ ]:
%sql
CREATE TABLE gold.fato_vendas (
    FK_TEMPO          DATE,
    FK_CLIENTE        BIGINT,
    FK_PRODUTO        BIGINT,
    NUMERO_PEDIDO     INT,
    STATUS_PEDIDO     STRING,
    QUANTIDADE        INT,
    PRECO_UNITARIO    DECIMAL(10,2),
    VALOR_TOTAL       DECIMAL(12,2)
) USING DELTA

In [ ]:
%sql
INSERT INTO gold.fato_vendas
SELECT dt.DATA                                   AS FK_TEMPO,
       dc.SK_CLIENTE                             AS FK_CLIENTE,
       dp.SK_PRODUTO                             AS FK_PRODUTO,
       p.CODIGO_PEDIDO                           AS NUMERO_PEDIDO,
       p.STATUS                                  AS STATUS_PEDIDO,
       i.QUANTIDADE,
       CAST(i.PRECO_UNITARIO AS DECIMAL(10,2))   AS PRECO_UNITARIO,
       CAST(i.QUANTIDADE * i.PRECO_UNITARIO AS DECIMAL(12,2)) AS VALOR_TOTAL
FROM item_pedido i
INNER JOIN pedido p          ON p.CODIGO_PEDIDO   = i.CODIGO_PEDIDO
INNER JOIN gold.dim_cliente dc ON dc.CODIGO_CLIENTE = p.CODIGO_CLIENTE
INNER JOIN gold.dim_produto dp ON dp.CODIGO_PRODUTO = i.CODIGO_PRODUTO
INNER JOIN gold.dim_tempo dt   ON dt.DATA           = to_date(p.DATA_PEDIDO)

In [ ]:
%sql
SELECT COUNT(*) AS ITENS_NO_FATO FROM gold.fato_vendas

## Consultas de negócio no esquema estrela

**Faturamento por categoria e ano** (sem pedidos cancelados):

In [ ]:
%sql
SELECT dp.CATEGORIA, dt.ANO, SUM(f.VALOR_TOTAL) AS FATURAMENTO, SUM(f.QUANTIDADE) AS ITENS
FROM gold.fato_vendas f
JOIN gold.dim_produto dp ON dp.SK_PRODUTO = f.FK_PRODUTO
JOIN gold.dim_tempo   dt ON dt.DATA       = f.FK_TEMPO
WHERE f.STATUS_PEDIDO <> 'CANCELADO'
GROUP BY dp.CATEGORIA, dt.ANO
ORDER BY dt.ANO, FATURAMENTO DESC

**Top 5 estados por faturamento:**

In [ ]:
%sql
SELECT dc.UNIDADE_FEDERATIVA, COUNT(DISTINCT f.NUMERO_PEDIDO) AS PEDIDOS, SUM(f.VALOR_TOTAL) AS FATURAMENTO
FROM gold.fato_vendas f
JOIN gold.dim_cliente dc ON dc.SK_CLIENTE = f.FK_CLIENTE
WHERE f.STATUS_PEDIDO <> 'CANCELADO'
GROUP BY dc.UNIDADE_FEDERATIVA
ORDER BY FATURAMENTO DESC
LIMIT 5

In [ ]:
%sql
SHOW TABLES IN gold